# Lesson 3 — Measurement, Noise, and Sampling

**Quantum Computing with Qiskit**

This notebook accompanies Lesson 3. Work through the cells in order. Every cell is meant to be run; modify the code freely and re-run to experiment.

**Topics covered:**
- Computational basis measurement and the Born rule
- Classical registers and bitstring ordering
- Multi-qubit measurement and `Statevector.probabilities_dict()`
- Partial measurement and mid-circuit measurement
- Shot-based sampling and convergence
- Shot noise: the binomial model and the 1/√N scaling
- Histogram analysis with `plot_histogram`
- Converting counts to probabilities

In [ ]:
!pip install qiskit qiskit-aer qiskit-ibm-runtime pylatexenc --quiet

In [ ]:
import qiskit
import qiskit_aer
import qiskit_ibm_runtime

print("Qiskit:             ", qiskit.__version__)
print("Qiskit Aer:         ", qiskit_aer.__version__)
print("Qiskit IBM Runtime: ", qiskit_ibm_runtime.__version__)

## 1. Computational Basis Measurement

The standard measurement in quantum computing asks a qubit whether it is $|0\rangle$ or $|1\rangle$.

The **Born rule** gives the probabilities: for $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$,

$$P(0) = |\alpha|^2, \qquad P(1) = |\beta|^2$$

After the measurement, the qubit **collapses** to the observed eigenstate — measuring again immediately returns the same result.

In Qiskit, `qc.measure(qubit, clbit)` connects a qubit to a classical bit.

In [ ]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(1, 1)   # 1 qubit, 1 classical bit
qc.h(0)                     # put qubit 0 in |+⟩
qc.measure(0, 0)            # measure qubit 0, store result in classical bit 0

qc.draw('mpl')

### Measuring all qubits at once

`measure_all()` is a convenience method that adds a classical register of the same size as the circuit and measures every qubit.

In [ ]:
qc = QuantumCircuit(3)
qc.h(0)
qc.cx(0, 1)
qc.cx(1, 2)
qc.measure_all()

qc.draw('mpl')

## 2. Classical Registers

A **classical register** is a named collection of classical bits. A circuit can have multiple classical registers, which is useful when tracking different groups of qubits separately.

When `get_counts()` is called on a circuit with multiple registers, the bitstrings concatenate all registers separated by a space, with the **last-added register on the left**.

**Bitstring ordering:** Qiskit uses little-endian convention — the **rightmost** character corresponds to qubit 0.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.circuit import QuantumRegister, ClassicalRegister

q = QuantumRegister(3, name='q')
c = ClassicalRegister(3, name='c')
qc = QuantumCircuit(q, c)

print(qc)

In [ ]:
from qiskit import transpile
from qiskit_aer import AerSimulator

q  = QuantumRegister(4, name='q')
c0 = ClassicalRegister(2, name='syndrome')   # error syndrome bits
c1 = ClassicalRegister(2, name='data')       # logical qubit outcomes

qc = QuantumCircuit(q, c0, c1)

# Put qubits 0 and 1 in superposition
qc.h([0, 1])
# Entangle qubits 2 and 3 with a Bell pair
qc.h(2)
qc.cx(2, 3)

qc.measure([0, 1], c0)    # measure qubits 0 and 1 into 'syndrome'
qc.measure([2, 3], c1)    # measure qubits 2 and 3 into 'data'

sim = AerSimulator()
counts = sim.run(transpile(qc, sim), shots=1024).result().get_counts()

# Qiskit concatenates registers in reverse order of addition.
# c1 ('data') was added last, so it appears on the LEFT.
# c0 ('syndrome') was added first, so it appears on the RIGHT.
# Bitstrings have the form 'data syndrome'
print("Sample counts (data syndrome):")
for bitstring, count in sorted(counts.items(), key=lambda x: -x[1])[:6]:
    print(f"  '{bitstring}': {count}")

qc.draw('mpl')

### Bitstring ordering demo

Apply X to qubit 0 only. The bitstring should be `'001'` (rightmost bit = qubit 0 = 1).

In [ ]:
qc_order = QuantumCircuit(3, 3)
qc_order.x(0)   # flip only qubit 0
qc_order.measure([0, 1, 2], [0, 1, 2])

sim = AerSimulator()
counts = sim.run(transpile(qc_order, sim), shots=128).result().get_counts()
print("Counts:", counts)
print("Expected: {'001': 128}  — qubit 0 is the rightmost bit")

## 3. Multi-Qubit Measurement

For an $n$-qubit system, a joint measurement returns one of $2^n$ possible bitstrings. `Statevector.probabilities_dict()` gives the exact theoretical probabilities without running any shots.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

qc = QuantumCircuit(3)
qc.h(0)
qc.h(1)
qc.h(2)

sv = Statevector.from_label('000').evolve(qc)
print("Exact probabilities:")
print(sv.probabilities_dict())

Three Hadamard gates produce a uniform superposition over all eight basis states, each with probability 1/8 = 0.125.

## 4. Partial Measurement and Mid-Circuit Measurement

### Partial measurement

Measuring fewer qubits than the circuit contains collapses only those qubits. The measured outcome conditions the state of the remaining qubits.

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

qc = QuantumCircuit(2, 1)   # 2 qubits, only 1 classical bit
qc.h(0)
qc.cx(0, 1)
qc.measure(0, 0)            # measure only qubit 0; qubit 1 is left alone

sim = AerSimulator()
counts = sim.run(transpile(qc, sim), shots=1024).result().get_counts()
print("Counts (only qubit 0 measured):", counts)
print("Expected: roughly {'0': 512, '1': 512}")

qc.draw('mpl')

### Mid-circuit measurement

Qiskit supports measuring qubits mid-circuit and then applying more gates. After a mid-circuit measurement, `qc.reset()` returns the qubit to $|0\rangle$ so it can be reused.

In [ ]:
qc = QuantumCircuit(2, 2)
qc.h(0)
qc.cx(0, 1)

# Measure qubit 0 mid-circuit
qc.measure(0, 0)

# Reset qubit 0 and reuse it
qc.reset(0)
qc.h(0)
qc.measure(0, 1)

qc.draw('mpl')

In [ ]:
sim = AerSimulator()
counts = sim.run(transpile(qc, sim), shots=1024).result().get_counts()
print("Mid-circuit measurement counts:", counts)
print("Classical bit 1 (leftmost) is from the second measurement: should be ~50/50")

## 5. Shot-Based Sampling

A quantum circuit returns one sample per execution. Running $N$ shots gives $N$ samples; the empirical frequencies approximate the true probabilities. The estimate improves as $N$ increases.

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

qc = QuantumCircuit(1, 1)
qc.h(0)
qc.measure(0, 0)

sim = AerSimulator()
t = transpile(qc, sim)

for shots in [10, 100, 1000, 10000]:
    counts = sim.run(t, shots=shots).result().get_counts()
    p0 = counts.get('0', 0) / shots
    print(f"{shots:6d} shots: P(0) ≈ {p0:.4f}")

## 6. Shot Noise and Precision

For a binary outcome with true probability $p$, the standard deviation of the estimated probability over $N$ shots is:

$$\sigma_{\hat{p}} = \sqrt{\frac{p(1-p)}{N}}$$

At $p = 0.5$ (worst case), $\sigma_{\hat{p}} = \frac{1}{2\sqrt{N}}$. Quadrupling $N$ halves the error.

The cell below runs the same circuit 500 times with 200 shots each and verifies that the observed spread matches the binomial prediction.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

qc = QuantumCircuit(1, 1)
qc.h(0)
qc.measure(0, 0)

sim = AerSimulator()
t = transpile(qc, sim)

shots = 200
n_repeats = 500

estimates = []
for _ in range(n_repeats):
    counts = sim.run(t, shots=shots).result().get_counts()
    estimates.append(counts.get('0', 0) / shots)

estimates = np.array(estimates)
p_true = 0.5

print(f"Mean estimate:           {estimates.mean():.4f}  (expected: {p_true:.4f})")
print(f"Observed std deviation:  {estimates.std():.4f}")
print(f"Predicted std deviation: {np.sqrt(p_true * (1 - p_true) / shots):.4f}")

The observed standard deviation should be very close to the theoretical prediction $\sqrt{0.5 \times 0.5 / 200} \approx 0.0354$.

### Shot count reference table

| Target precision ε | Required shots |
|---|---|
| 5% | 100 |
| 2% | 625 |
| 1% | 2500 |
| 0.1% | 250 000 |

## 7. Histogram Analysis

`plot_histogram` visualizes the count dictionary returned by `get_counts()`.

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

qc = QuantumCircuit(3, 3)
qc.h(0)
qc.h(1)
qc.h(2)
qc.measure([0, 1, 2], [0, 1, 2])

sim = AerSimulator()
counts = sim.run(transpile(qc, sim), shots=4096).result().get_counts()

plot_histogram(counts, title='Uniform superposition over 3 qubits')

### Overlaying ideal and noisy results

Pass a list of two count dictionaries and a `legend` argument to overlay results side by side. `FakeBrisbane` is a local mock of a real IBM device that includes a hardware noise model.

In [ ]:
from qiskit_ibm_runtime.fake_provider import FakeBrisbane
from qiskit import transpile

# Ideal counts from AerSimulator (already computed above)
ideal_counts = sim.run(transpile(qc, sim), shots=4096).result().get_counts()

# Noisy counts from FakeBrisbane (includes hardware noise model)
fake = FakeBrisbane()
noisy_counts = fake.run(transpile(qc, fake), shots=4096).result().get_counts()

plot_histogram(
    [ideal_counts, noisy_counts],
    legend=['Ideal', 'Noisy (FakeBrisbane)'],
    title='Ideal vs noisy uniform superposition'
)

The noisy histogram shows slightly unequal bar heights because gate errors and readout errors shift probabilities away from the uniform distribution.

### Converting counts to probabilities

In [ ]:
total = sum(counts.values())
probs = {bitstring: count / total for bitstring, count in counts.items()}
print("Probabilities:")
for b, p in sorted(probs.items()):
    print(f"  {b}: {p:.4f}")

In [ ]:
# Zero-fill missing bitstrings (outcomes that never occurred but are possible)
n_qubits = 3
all_bitstrings = [format(i, f'0{n_qubits}b') for i in range(2**n_qubits)]
probs_full = {b: counts.get(b, 0) / total for b in all_bitstrings}

print("Full probability table:")
for b, p in sorted(probs_full.items()):
    print(f"  |{b}⟩: {p:.4f}")

## 8. Putting It Together: Shot Count Comparison

Running the same circuit at different shot counts shows the trade-off between precision and computation time.

In [ ]:
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

qc = QuantumCircuit(2, 2)
qc.h(0)
qc.cx(0, 1)
qc.measure([0, 1], [0, 1])

sim = AerSimulator()
t = transpile(qc, sim)
shot_counts = [64, 512, 4096]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, shots in zip(axes, shot_counts):
    counts = sim.run(t, shots=shots).result().get_counts()
    plot_histogram(counts, ax=ax, title=f'{shots} shots')

plt.suptitle('Bell State Measurement: Shot Count Comparison', fontsize=13)
plt.tight_layout()
plt.show()

With 64 shots the two bars are noticeably unequal. With 4096 shots they converge to near-equal height, reflecting the theoretical 50/50 distribution.

## 9. Exercises

Work through these exercises to consolidate what you learned in this lesson.

### Exercise 1: Born rule verification

Build a single-qubit circuit that applies the rotation gate `qc.ry(theta, 0)` for $\theta = \pi/3$, followed by a measurement.

The Born rule predicts $P(0) = \cos^2(\theta/2)$. With $\theta = \pi/3$, the predicted $P(0)$ is $\cos^2(\pi/6) = 3/4 = 0.75$.

Run the circuit with 4096 shots and verify that the measured frequency is close to 0.75. Also use `Statevector.probabilities_dict()` to confirm the exact value.

In [ ]:
# Exercise 1: your solution here
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector

theta = np.pi / 3

qc_ex1 = QuantumCircuit(1, 1)
# TODO: apply Ry(theta) and measure

# Shot-based estimate
sim = AerSimulator()
# TODO: run and print P(0)

# Exact prediction
# TODO: use Statevector.probabilities_dict() to confirm

### Exercise 2: Multiple classical registers

Build a 4-qubit circuit with two 2-bit classical registers named `'alice'` and `'bob'`. Apply H to all four qubits, then measure qubits 0 and 1 into `'alice'` and qubits 2 and 3 into `'bob'`. Run with 2048 shots and print the counts.

Identify the format of the bitstrings in `get_counts()`. Which register appears on the left?

In [ ]:
# Exercise 2: your solution here
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator

# TODO: build the circuit with two named classical registers
# TODO: run and print the counts
# TODO: comment on the bitstring format

### Exercise 3: Shot noise scaling

For the H-gate circuit (50/50 outcome), run 200 independent trials at each of the following shot counts: 25, 100, 400, 1600. For each shot count, compute the standard deviation of the 200 estimated $P(0)$ values and compare it to the theoretical prediction $1/(2\sqrt{N})$.

Plot the observed standard deviation against $1/\sqrt{N}$ to verify the scaling.

In [ ]:
# Exercise 3: your solution here
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

qc_ex3 = QuantumCircuit(1, 1)
qc_ex3.h(0)
qc_ex3.measure(0, 0)

sim = AerSimulator()
t = transpile(qc_ex3, sim)

shot_values = [25, 100, 400, 1600]
n_trials = 200

# TODO: for each shot count, run n_trials experiments and record std deviation
# TODO: plot observed std vs 1/(2*sqrt(N))

### Exercise 4: Ideal vs noisy GHZ state

Build a 3-qubit GHZ state circuit (H on qubit 0, CNOT 0→1, CNOT 1→2, measure all). The ideal distribution should show only `'000'` and `'111'`, each with probability 0.5.

Run both on `AerSimulator` (ideal) and `FakeBrisbane` (noisy) with 4096 shots. Use `plot_histogram` to overlay the two results. Quantify the noise: what fraction of outcomes fall outside `{'000', '111'}`?

In [ ]:
# Exercise 4: your solution here
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime.fake_provider import FakeBrisbane
from qiskit.visualization import plot_histogram

qc_ghz = QuantumCircuit(3, 3)
# TODO: build the GHZ circuit

# TODO: run on AerSimulator and FakeBrisbane
# TODO: overlay the two histograms
# TODO: compute and print the fraction of error outcomes

## Summary

In this lesson you:

- Applied the Born rule: probability of outcome 0 is $|\alpha|^2$, outcome 1 is $|\beta|^2$.
- Used `qc.measure()`, `qc.measure_all()`, and multiple named classical registers.
- Confirmed Qiskit's little-endian bitstring convention: qubit 0 is the rightmost character.
- Performed partial measurement and mid-circuit measurement with `qc.reset()`.
- Verified that shot noise follows the binomial model: $\sigma_{\hat{p}} = \sqrt{p(1-p)/N}$.
- Used `plot_histogram` to visualize count dictionaries and overlay ideal vs noisy results.
- Converted raw counts to probabilities and zero-filled missing bitstrings.

**Lesson 4** introduces entanglement: the four Bell states, correlation measurements, quantum teleportation, and superdense coding.